# 07. Evaluación predictiva temporal de 2025

Este cuaderno abre la partición 2025 después de cargar el protocolo y los modelos congelados con 2019–2024. No reajusta ninguna transformación ni modelo: genera las predicciones externas, calcula las métricas y sensibilidades, cuantifica la incertidumbre de las mejoras y guarda los productos finales de evaluación.


In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import load
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)


## 1. Protocolo, modelos y partición de prueba


In [ ]:
# Localizar la raíz y cargar los artefactos congelados

ruta_actual = Path.cwd().resolve()

raiz_proyecto = next(
    (
        ruta
        for ruta in [ruta_actual, *ruta_actual.parents]
        if (ruta / "pyproject.toml").is_file()
    ),
    None,
)

if raiz_proyecto is None:
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto. "
        "Abre VS Code o Jupyter desde el repositorio."
    )

sys.path.insert(0, str(raiz_proyecto))

ruta_config = (
    raiz_proyecto
    / "configs"
    / "protocolo_ml_pre2025.json"
)

ruta_prueba = (
    raiz_proyecto
    / "data"
    / "processed"
    / "geminidas_modelado_v2"
    / "geminidas_modelado_prueba_2025_v2.parquet"
)

for ruta in [ruta_config, ruta_prueba]:
    if not ruta.is_file():
        raise FileNotFoundError(
            f"No se encontró {ruta.relative_to(raiz_proyecto)}."
        )

hash_protocolo_inicial = hashlib.sha256(
    ruta_config.read_bytes()
).hexdigest()

with ruta_config.open("r", encoding="utf-8") as archivo:
    protocolo = json.load(archivo)

ruta_modelos_relativa = Path(
    str(protocolo["archivo_modelos"]).replace("\\", "/")
)

ruta_modelos = raiz_proyecto / ruta_modelos_relativa

if not ruta_modelos.is_file():
    raise FileNotFoundError(
        f"No se encontró {ruta_modelos_relativa}. "
        "Ejecuta primero el notebook 06."
    )

hash_modelos_inicial = hashlib.sha256(
    ruta_modelos.read_bytes()
).hexdigest()

if hash_modelos_inicial != protocolo["sha256_modelos"]:
    raise ValueError(
        "El archivo de modelos no coincide con el SHA-256 "
        "registrado en el protocolo pre-2025."
    )

if protocolo["prueba_reservada"] != 2025:
    raise ValueError("El protocolo no reserva 2025 como prueba.")

if protocolo["reajustar_con_2025"] is not False:
    raise ValueError("El protocolo permite reajustar con 2025.")

modelos = load(ruta_modelos)

claves_esperadas = {
    "control",
    "local_cuadratico",
    "pca_2_cuadratico",
    "pca_global",
    "variables_cinematica",
}

if not claves_esperadas.issubset(modelos):
    raise KeyError(
        "El paquete de modelos no contiene todos los objetos esperados."
    )

for nombre in [
    "control",
    "local_cuadratico",
    "pca_2_cuadratico",
]:
    clases = modelos[nombre].named_steps["clasificador"].classes_

    if not np.array_equal(clases, [0, 1]):
        raise ValueError(
            f"Las clases del modelo {nombre} no son [0, 1]."
        )

df_2025 = pd.read_parquet(ruta_prueba)

variables_modelo = (
    protocolo["variables_fase"]
    + protocolo["variables_calidad"]
    + protocolo["variables_cinematica"]
)

columnas_necesarias = [
    "id_fila_origen",
    "Beginning (UTC Time)",
    "Participating (stations)",
    "source_year",
    "particion_modelado",
    "usar_modelo_ml",
    "objetivo_ml",
    "grupo_brillo_ml",
    "pertenece_muestra_estricta",
    "magnitud_analisis",
    *variables_modelo,
]

faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in df_2025.columns
]

if faltantes:
    raise KeyError(
        f"Faltan columnas necesarias en la prueba 2025: {faltantes}"
    )

if not df_2025["id_fila_origen"].is_unique:
    raise ValueError("La base 2025 contiene IDs repetidos.")

if set(df_2025["source_year"].unique()) != {2025}:
    raise ValueError("La base de prueba contiene años distintos de 2025.")

if set(df_2025["particion_modelado"].unique()) != {"prueba"}:
    raise ValueError("La partición de 2025 no está marcada como prueba.")

df_2025_ml = df_2025.loc[
    df_2025["usar_modelo_ml"]
].copy()

if set(df_2025_ml["objetivo_ml"].dropna().astype(int).unique()) != {0, 1}:
    raise ValueError("La prueba no contiene las dos clases binarias.")

if not df_2025_ml["objetivo_ml"].notna().all():
    raise ValueError("Hay objetivos faltantes entre las filas de ML.")

if not np.isfinite(
    df_2025_ml[variables_modelo].to_numpy(dtype=float)
).all():
    raise ValueError(
        "Los predictores de la prueba contienen valores no finitos."
    )

limite_brillante = float(protocolo["limite_brillante"])
limite_tenue = float(protocolo["limite_tenue"])

objetivo_2025 = df_2025_ml["objetivo_ml"].astype(int)

if not df_2025_ml.loc[
    objetivo_2025 == 0,
    "magnitud_analisis",
].le(limite_brillante).all():
    raise ValueError("La clase brillante no respeta el límite pre-2025.")

if not df_2025_ml.loc[
    objetivo_2025 == 1,
    "magnitud_analisis",
].ge(limite_tenue).all():
    raise ValueError("La clase tenue no respeta el límite pre-2025.")

print("Protocolo y modelos verificados.")
print("Trayectorias de 2025:", len(df_2025))
print("Trayectorias para clasificación:", len(df_2025_ml))
print(
    "Muestra estricta para clasificación:",
    int(df_2025_ml["pertenece_muestra_estricta"].sum()),
)
print("Modelo principal:", protocolo["modelo_principal"])


## 2. Predicciones de los modelos congelados


In [ ]:
# Aplicar los tres modelos sin reajustar nada con 2025

if (
    list(modelos["variables_cinematica"])
    != protocolo["variables_cinematica"]
):
    raise ValueError(
        "Las variables cinemáticas del modelo no coinciden con el protocolo."
    )

if modelos["pca_global"].components_.shape[0] < 2:
    raise ValueError("El PCA guardado no contiene al menos dos componentes.")

datos_pca_2025 = df_2025_ml.copy()

coordenadas_pca_2025 = modelos["pca_global"].transform(
    datos_pca_2025[modelos["variables_cinematica"]]
)

datos_pca_2025["PC1"] = coordenadas_pca_2025[:, 0]
datos_pca_2025["PC2"] = coordenadas_pca_2025[:, 1]

modelos_prueba = {
    "Control": ("control", df_2025_ml),
    "Local cuadrático": (
        "local_cuadratico",
        df_2025_ml,
    ),
    "PCA 2 cuadrático": (
        "pca_2_cuadratico",
        datos_pca_2025,
    ),
}

umbral = float(protocolo["umbral_clasificacion"])
tablas_predicciones = []

for nombre, (clave, datos) in modelos_prueba.items():
    probabilidades = modelos[clave].predict_proba(datos)[:, 1]

    tabla = df_2025_ml[
        [
            "id_fila_origen",
            "Beginning (UTC Time)",
            "source_year",
            "objetivo_ml",
            "pertenece_muestra_estricta",
            "magnitud_analisis",
        ]
    ].copy()

    tabla = tabla.rename(
        columns={"objetivo_ml": "objetivo_real"}
    )

    tabla["modelo"] = nombre
    tabla["probabilidad_tenue"] = probabilidades
    tabla["prediccion"] = (
        probabilidades >= umbral
    ).astype(int)

    tablas_predicciones.append(tabla)

predicciones_2025 = pd.concat(
    tablas_predicciones,
    ignore_index=True,
)

ids_esperados = set(df_2025_ml["id_fila_origen"])

if predicciones_2025.duplicated(
    ["modelo", "id_fila_origen"]
).any():
    raise ValueError(
        "Hay predicciones duplicadas por modelo e identificador."
    )

for nombre, grupo in predicciones_2025.groupby("modelo"):
    if set(grupo["id_fila_origen"]) != ids_esperados:
        raise ValueError(
            f"El modelo {nombre} no evaluó exactamente las mismas filas."
        )

    if not np.isfinite(
        grupo["probabilidad_tenue"].to_numpy(dtype=float)
    ).all():
        raise ValueError(
            f"El modelo {nombre} produjo probabilidades no finitas."
        )

    if not grupo["probabilidad_tenue"].between(0, 1).all():
        raise ValueError(
            f"El modelo {nombre} produjo probabilidades fuera de [0, 1]."
        )

ruta_tablas = raiz_proyecto / "reports" / "tables"
ruta_figuras = raiz_proyecto / "reports" / "figures"

ruta_tablas.mkdir(parents=True, exist_ok=True)
ruta_figuras.mkdir(parents=True, exist_ok=True)

ruta_predicciones = (
    ruta_tablas
    / "predicciones_2025.parquet"
)

predicciones_2025.to_parquet(
    ruta_predicciones,
    index=False,
)

display(
    predicciones_2025.groupby("modelo")
    .size()
    .rename("trayectorias")
    .to_frame()
)


## 3. Métricas y análisis de sensibilidad


In [ ]:
# Calcular las métricas sobre las mismas filas para cada modelo

corte_extremos = float(
    protocolo["sensibilidad_fotometrica"]["corte"]
)

filas_metricas = []
filas_confusion = []

for nombre, datos_modelo in predicciones_2025.groupby("modelo"):
    conjuntos = {
        "Estándar": datos_modelo,
        "Estricta": datos_modelo.loc[
            datos_modelo["pertenece_muestra_estricta"]
        ],
        "Sin extremos": datos_modelo.loc[
            datos_modelo["magnitud_analisis"]
            >= corte_extremos
        ],
    }

    for muestra, grupo in conjuntos.items():
        y = grupo["objetivo_real"].astype(int)
        p = grupo["probabilidad_tenue"].to_numpy(dtype=float)
        pred = grupo["prediccion"].astype(int)

        if set(y.unique()) != {0, 1}:
            raise ValueError(
                f"La muestra {muestra} del modelo {nombre} "
                "no contiene ambas clases."
            )

        tn, fp, fn, tp = confusion_matrix(
            y,
            pred,
            labels=[0, 1],
        ).ravel()

        filas_metricas.append(
            {
                "modelo": nombre,
                "muestra": muestra,
                "trayectorias": len(grupo),
                "brillantes": int((y == 0).sum()),
                "tenues": int((y == 1).sum()),
                "prevalencia_tenues": float(y.mean()),
                "roc_auc": roc_auc_score(y, p),
                "average_precision": average_precision_score(y, p),
                "exactitud": accuracy_score(y, pred),
                "exactitud_balanceada": balanced_accuracy_score(y, pred),
                "sensibilidad_tenues": recall_score(
                    y,
                    pred,
                    pos_label=1,
                ),
                "especificidad_brillantes": recall_score(
                    y,
                    pred,
                    pos_label=0,
                ),
                "precision_tenues": precision_score(
                    y,
                    pred,
                    zero_division=0,
                ),
                "f1_tenues": f1_score(
                    y,
                    pred,
                    zero_division=0,
                ),
            }
        )

        filas_confusion.extend(
            [
                {
                    "modelo": nombre,
                    "muestra": muestra,
                    "real": "Brillante",
                    "predicha": "Brillante",
                    "conteo": int(tn),
                },
                {
                    "modelo": nombre,
                    "muestra": muestra,
                    "real": "Brillante",
                    "predicha": "Tenue",
                    "conteo": int(fp),
                },
                {
                    "modelo": nombre,
                    "muestra": muestra,
                    "real": "Tenue",
                    "predicha": "Brillante",
                    "conteo": int(fn),
                },
                {
                    "modelo": nombre,
                    "muestra": muestra,
                    "real": "Tenue",
                    "predicha": "Tenue",
                    "conteo": int(tp),
                },
            ]
        )

metricas_2025 = pd.DataFrame(filas_metricas)
matrices_confusion_2025 = pd.DataFrame(filas_confusion)

ruta_metricas = ruta_tablas / "metricas_2025.csv"
ruta_confusion = ruta_tablas / "matrices_confusion_2025.csv"

metricas_2025.to_csv(
    ruta_metricas,
    index=False,
)

matrices_confusion_2025.to_csv(
    ruta_confusion,
    index=False,
)

tabla_auc = metricas_2025.pivot(
    index="muestra",
    columns="modelo",
    values="roc_auc",
)

tabla_auc["mejora_local"] = (
    tabla_auc["Local cuadrático"]
    - tabla_auc["Control"]
)

tabla_auc["mejora_pca2"] = (
    tabla_auc["PCA 2 cuadrático"]
    - tabla_auc["Control"]
)

display(
    metricas_2025[
        [
            "modelo",
            "muestra",
            "trayectorias",
            "roc_auc",
            "average_precision",
            "exactitud_balanceada",
        ]
    ].round(4)
)

print("\nDiferencias de ROC-AUC frente al control:")
display(tabla_auc.round(4))


## 4. Dependencia temporal y observacional


In [ ]:
# Contar noches y combinaciones de estaciones presentes en la prueba

fechas_utc = pd.to_datetime(
    df_2025_ml["Beginning (UTC Time)"],
    errors="coerce",
    utc=True,
)

if fechas_utc.isna().any():
    raise ValueError(
        "No fue posible interpretar todas las fechas UTC de 2025."
    )

def normalizar_estaciones(valor):
    estaciones = [
        estacion.strip()
        for estacion in str(valor).split(",")
        if estacion.strip()
    ]

    return "|".join(sorted(set(estaciones)))


metadata_2025 = df_2025_ml[
    [
        "id_fila_origen",
        "objetivo_ml",
        "pertenece_muestra_estricta",
        "magnitud_analisis",
        "Participating (stations)",
    ]
].copy()

metadata_2025["fecha_utc"] = fechas_utc.dt.date
metadata_2025["combinacion_estaciones"] = (
    metadata_2025["Participating (stations)"]
    .map(normalizar_estaciones)
)

conteo_noches_2025 = (
    metadata_2025.groupby("fecha_utc")
    .agg(
        trayectorias=("id_fila_origen", "size"),
        brillantes=("objetivo_ml", lambda s: int((s == 0).sum())),
        tenues=("objetivo_ml", lambda s: int((s == 1).sum())),
    )
    .reset_index()
)

resumen_dependencia_2025 = pd.DataFrame(
    [
        {
            "trayectorias_ml": len(metadata_2025),
            "noches_utc": metadata_2025["fecha_utc"].nunique(),
            "combinaciones_estaciones": (
                metadata_2025[
                    "combinacion_estaciones"
                ].nunique()
            ),
            "nota": (
                "Las filas comparten noches y estaciones; "
                "no deben interpretarse como unidades completamente independientes."
            ),
        }
    ]
)

ruta_noches = ruta_tablas / "conteo_por_noche_2025.csv"
ruta_dependencia = ruta_tablas / "resumen_dependencia_2025.csv"

conteo_noches_2025.to_csv(
    ruta_noches,
    index=False,
)

resumen_dependencia_2025.to_csv(
    ruta_dependencia,
    index=False,
)

display(resumen_dependencia_2025)
display(conteo_noches_2025)

if metadata_2025["fecha_utc"].nunique() < 15:
    print(
        "Advertencia: hay pocas noches independientes para justificar "
        "un bootstrap nocturno estable. Los intervalos siguientes "
        "usan remuestreo pareado por observación y no modelan "
        "completamente la dependencia nocturna."
    )


## 5. Incertidumbre de las mejoras


In [ ]:
# Bootstrap pareado y estratificado por clase para las diferencias de rendimiento

probabilidades_por_modelo = (
    predicciones_2025.pivot(
        index="id_fila_origen",
        columns="modelo",
        values="probabilidad_tenue",
    )
)

tabla_bootstrap = (
    metadata_2025
    .set_index("id_fila_origen")
    .join(
        probabilidades_por_modelo,
        how="inner",
        validate="one_to_one",
    )
)

if len(tabla_bootstrap) != len(df_2025_ml):
    raise ValueError(
        "La tabla del bootstrap no contiene todas las filas de prueba."
    )

comparaciones_bootstrap = {
    "Local cuadrático - Control": (
        "Local cuadrático",
        "Control",
    ),
    "PCA 2 cuadrático - Control": (
        "PCA 2 cuadrático",
        "Control",
    ),
}

def bootstrap_pareado_estratificado(
    tabla,
    nombre_muestra,
    repeticiones=2000,
    semilla=20261126,
):
    y = tabla["objetivo_ml"].astype(int).to_numpy()

    indices_clase_0 = np.flatnonzero(y == 0)
    indices_clase_1 = np.flatnonzero(y == 1)

    if len(indices_clase_0) == 0 or len(indices_clase_1) == 0:
        raise ValueError(
            f"La muestra {nombre_muestra} no contiene ambas clases."
        )

    rng = np.random.default_rng(semilla)
    filas = []

    for nombre_comparacion, (
        modelo_alternativo,
        modelo_control,
    ) in comparaciones_bootstrap.items():
        p_alternativo = tabla[
            modelo_alternativo
        ].to_numpy(dtype=float)

        p_control = tabla[
            modelo_control
        ].to_numpy(dtype=float)

        auc_observada = (
            roc_auc_score(y, p_alternativo)
            - roc_auc_score(y, p_control)
        )

        ap_observada = (
            average_precision_score(y, p_alternativo)
            - average_precision_score(y, p_control)
        )

        diferencias_auc = np.empty(
            repeticiones,
            dtype=float,
        )

        diferencias_ap = np.empty(
            repeticiones,
            dtype=float,
        )

        for repeticion in range(repeticiones):
            muestra_0 = rng.choice(
                indices_clase_0,
                size=len(indices_clase_0),
                replace=True,
            )

            muestra_1 = rng.choice(
                indices_clase_1,
                size=len(indices_clase_1),
                replace=True,
            )

            indices = np.concatenate(
                [muestra_0, muestra_1]
            )

            y_boot = y[indices]

            diferencias_auc[repeticion] = (
                roc_auc_score(
                    y_boot,
                    p_alternativo[indices],
                )
                - roc_auc_score(
                    y_boot,
                    p_control[indices],
                )
            )

            diferencias_ap[repeticion] = (
                average_precision_score(
                    y_boot,
                    p_alternativo[indices],
                )
                - average_precision_score(
                    y_boot,
                    p_control[indices],
                )
            )

        for metrica, observada, distribucion in [
            (
                "roc_auc",
                auc_observada,
                diferencias_auc,
            ),
            (
                "average_precision",
                ap_observada,
                diferencias_ap,
            ),
        ]:
            limite_inferior, limite_superior = np.percentile(
                distribucion,
                [2.5, 97.5],
            )

            filas.append(
                {
                    "muestra": nombre_muestra,
                    "comparacion": nombre_comparacion,
                    "metrica": metrica,
                    "diferencia_observada": observada,
                    "ic95_inferior": limite_inferior,
                    "ic95_superior": limite_superior,
                    "repeticiones": repeticiones,
                    "metodo": (
                        "bootstrap pareado estratificado por clase, "
                        "por observación"
                    ),
                }
            )

    return pd.DataFrame(filas)


mascaras_bootstrap = {
    "Estándar": pd.Series(
        True,
        index=tabla_bootstrap.index,
    ),
    "Estricta": tabla_bootstrap[
        "pertenece_muestra_estricta"
    ].astype(bool),
    "Sin extremos": (
        tabla_bootstrap["magnitud_analisis"]
        >= corte_extremos
    ),
}

tablas_bootstrap = []

for numero, (
    nombre_muestra,
    mascara,
) in enumerate(mascaras_bootstrap.items()):
    tablas_bootstrap.append(
        bootstrap_pareado_estratificado(
            tabla_bootstrap.loc[mascara].copy(),
            nombre_muestra,
            repeticiones=2000,
            semilla=20261126 + numero,
        )
    )

bootstrap_2025 = pd.concat(
    tablas_bootstrap,
    ignore_index=True,
)

ruta_bootstrap = (
    ruta_tablas
    / "bootstrap_diferencias_2025.csv"
)

bootstrap_2025.to_csv(
    ruta_bootstrap,
    index=False,
)

display(bootstrap_2025.round(4))


## 6. Coeficientes del modelo principal


In [ ]:
# Exportar los coeficientes estandarizados del modelo local cuadrático final

modelo_local_final = modelos["local_cuadratico"]
clasificador_local = modelo_local_final.named_steps["clasificador"]

nombres_terminos = (
    [
        protocolo["variables_fase"][0],
        f"{protocolo['variables_fase'][0]}^2",
    ]
    + protocolo["variables_calidad"]
    + protocolo["variables_cinematica"]
    + [
        f"{variable}^2"
        for variable in protocolo["variables_cinematica"]
    ]
)

coeficientes = clasificador_local.coef_[0]

if len(coeficientes) != len(nombres_terminos):
    raise ValueError(
        "No coincide el número de coeficientes del modelo principal."
    )

tipos_terminos = (
    ["fase", "fase"]
    + ["calidad"] * len(protocolo["variables_calidad"])
    + ["cinematica_lineal"] * len(protocolo["variables_cinematica"])
    + ["cinematica_cuadratica"] * len(protocolo["variables_cinematica"])
)

coeficientes_modelo_local = pd.DataFrame(
    {
        "termino": nombres_terminos,
        "tipo": tipos_terminos,
        "coeficiente_estandarizado": coeficientes,
    }
)

coeficientes_modelo_local["intercepto_modelo"] = float(
    clasificador_local.intercept_[0]
)

ruta_coeficientes = (
    ruta_tablas
    / "coeficientes_modelo_local_final_pre2025.csv"
)

coeficientes_modelo_local.to_csv(
    ruta_coeficientes,
    index=False,
)

display(
    coeficientes_modelo_local.loc[
        coeficientes_modelo_local["tipo"].str.startswith(
            "cinematica"
        )
    ].round(4)
)

print(
    "Los coeficientes corresponden a predictores transformados "
    "y estandarizados; los términos lineales y cuadrados deben "
    "interpretarse conjuntamente."
)


## 7. Figuras de evaluación externa


In [ ]:
# Curvas ROC y precisión-exhaustividad de la muestra estándar

orden_modelos = [
    "Control",
    "Local cuadrático",
    "PCA 2 cuadrático",
]

base_estandar = predicciones_2025.copy()

fig, ax = plt.subplots(figsize=(7, 5))

for nombre in orden_modelos:
    datos = base_estandar.loc[
        base_estandar["modelo"] == nombre
    ]

    y = datos["objetivo_real"].astype(int)
    p = datos["probabilidad_tenue"]

    fpr, tpr, _ = roc_curve(y, p)
    auc = roc_auc_score(y, p)

    ax.plot(
        fpr,
        tpr,
        label=f"{nombre} (AUC={auc:.3f})",
    )

ax.plot(
    [0, 1],
    [0, 1],
    "--",
    alpha=0.6,
)

ax.set(
    title="Curvas ROC — Gemínidas 2025",
    xlabel="Tasa de falsos positivos",
    ylabel="Sensibilidad",
    xlim=(0, 1),
    ylim=(0, 1),
)

ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()

ruta_roc_pdf = ruta_figuras / "roc_2025.pdf"
ruta_roc_png = ruta_figuras / "roc_2025.png"

fig.savefig(
    ruta_roc_pdf,
    bbox_inches="tight",
)

fig.savefig(
    ruta_roc_png,
    dpi=300,
    bbox_inches="tight",
)

plt.show()


fig, ax = plt.subplots(figsize=(7, 5))

for nombre in orden_modelos:
    datos = base_estandar.loc[
        base_estandar["modelo"] == nombre
    ]

    y = datos["objetivo_real"].astype(int)
    p = datos["probabilidad_tenue"]

    precision, sensibilidad, _ = (
        precision_recall_curve(y, p)
    )

    ap = average_precision_score(y, p)

    ax.plot(
        sensibilidad,
        precision,
        label=f"{nombre} (AP={ap:.3f})",
    )

prevalencia = float(
    df_2025_ml["objetivo_ml"].astype(int).mean()
)

ax.axhline(
    prevalencia,
    linestyle="--",
    label=f"Prevalencia tenue ({prevalencia:.3f})",
)

ax.set(
    title="Precisión-exhaustividad — Gemínidas 2025",
    xlabel="Sensibilidad",
    ylabel="Precisión",
    xlim=(0, 1),
    ylim=(0, 1),
)

ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()

ruta_pr_pdf = (
    ruta_figuras
    / "precision_exhaustividad_2025.pdf"
)

ruta_pr_png = (
    ruta_figuras
    / "precision_exhaustividad_2025.png"
)

fig.savefig(
    ruta_pr_pdf,
    bbox_inches="tight",
)

fig.savefig(
    ruta_pr_png,
    dpi=300,
    bbox_inches="tight",
)

plt.show()


In [ ]:
# Matrices de confusión del control y del modelo principal

rutas_confusion_figuras = []

for nombre in ["Control", "Local cuadrático"]:
    datos = predicciones_2025.loc[
        predicciones_2025["modelo"] == nombre
    ]

    y = datos["objetivo_real"].astype(int)
    pred = datos["prediccion"].astype(int)

    matriz = confusion_matrix(
        y,
        pred,
        labels=[0, 1],
    )

    fig, ax = plt.subplots(figsize=(5, 4))

    ConfusionMatrixDisplay(
        confusion_matrix=matriz,
        display_labels=["Brillante", "Tenue"],
    ).plot(
        ax=ax,
        values_format="d",
        colorbar=False,
    )

    ax.set_title(
        f"Matriz de confusión — {nombre} — 2025"
    )

    fig.tight_layout()

    nombre_archivo = (
        "confusion_control_2025"
        if nombre == "Control"
        else "confusion_local_cuadratico_2025"
    )

    ruta_pdf = ruta_figuras / f"{nombre_archivo}.pdf"
    ruta_png = ruta_figuras / f"{nombre_archivo}.png"

    fig.savefig(
        ruta_pdf,
        bbox_inches="tight",
    )

    fig.savefig(
        ruta_png,
        dpi=300,
        bbox_inches="tight",
    )

    rutas_confusion_figuras.extend(
        [ruta_pdf, ruta_png]
    )

    plt.show()


## 8. Verificación final y manifiesto


In [ ]:
# Comprobar que la evaluación no modificó los artefactos pre-2025

hash_protocolo_final = hashlib.sha256(
    ruta_config.read_bytes()
).hexdigest()

hash_modelos_final = hashlib.sha256(
    ruta_modelos.read_bytes()
).hexdigest()

if hash_protocolo_final != hash_protocolo_inicial:
    raise RuntimeError(
        "El notebook 07 modificó el protocolo pre-2025."
    )

if hash_modelos_final != hash_modelos_inicial:
    raise RuntimeError(
        "El notebook 07 modificó el archivo de modelos pre-2025."
    )

rutas_resultados = [
    ruta_predicciones,
    ruta_metricas,
    ruta_confusion,
    ruta_noches,
    ruta_dependencia,
    ruta_bootstrap,
    ruta_coeficientes,
    ruta_roc_pdf,
    ruta_roc_png,
    ruta_pr_pdf,
    ruta_pr_png,
    *rutas_confusion_figuras,
]

faltantes_resultados = [
    str(ruta.relative_to(raiz_proyecto))
    for ruta in rutas_resultados
    if not ruta.is_file()
]

if faltantes_resultados:
    raise FileNotFoundError(
        "Faltan productos finales de evaluación:\n"
        + "\n".join(faltantes_resultados)
    )

def sha256_archivo(ruta):
    sha = hashlib.sha256()

    with ruta.open("rb") as archivo:
        for bloque in iter(
            lambda: archivo.read(1024 * 1024),
            b"",
        ):
            sha.update(bloque)

    return sha.hexdigest()


manifiesto_evaluacion = pd.DataFrame(
    [
        {
            "archivo": str(
                ruta.relative_to(raiz_proyecto)
            ).replace("\\", "/"),
            "tamano_bytes": ruta.stat().st_size,
            "sha256": sha256_archivo(ruta),
        }
        for ruta in rutas_resultados
    ]
)

ruta_manifiesto = (
    ruta_tablas
    / "manifiesto_evaluacion_2025.csv"
)

manifiesto_evaluacion.to_csv(
    ruta_manifiesto,
    index=False,
)

print("Evaluación 2025 completada.")
print("El protocolo y los modelos pre-2025 permanecen sin cambios.")
print(
    "Manifiesto:",
    ruta_manifiesto.relative_to(raiz_proyecto),
)

display(manifiesto_evaluacion)
